# 10 Final holdout (locked)

**What this does.** Holds the final check, out of reach of development: the 160 holdout companies (01's split) and
Barbara's 24 direct-peer pairs whose comp is in the universe (cohort `final_holdout_24`; all 24 comps are holdout
companies). Nothing in 01-09 is chosen with them.

**By default this notebook only takes inventory and checks that nothing leaked.** The results section runs only
when `FINAL_CHECK = True`; set it when the team decides the method is final (Michael's call), run once, and record
the run in PROJECT_STATE.md.

**Changelog**
- 2026-11-19: created. `final_check.py` = Fiona's `direct_peer_pairs.py standardize` (11-18, without the X-learner
  inputs, which were deleted) + the current B3 lists and valuation on the holdout. It reproduces the 11-18
  standardized pairs exactly. v1's holdout scripts (v0 routes, 10-28 to 11-10) stay in v1 as the record; their
  saved results are in `outputs-local/v1a/10_final_holdout/v1_holdout_runs/`.

In [1]:
FINAL_CHECK = False   # the team's decision; run once

In [2]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "v2" / "current.toml").exists())
sys.path.insert(0, str(ROOT / "v2" / "common"))
import paths, nbtools as nb
paths.setup()
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 80)
cur = paths.current()
checks = nb.Checks()

In [3]:
import json, cohorts, final_check as fc
split = pd.read_csv(paths.path(cur["universe"]["split"]), dtype=str)
training = set(split.loc[split["sample"] == "training", "IQ_COMPANY_ID"])
holdout = set(split.loc[split["sample"] == "holdout", "IQ_COMPANY_ID"])
print(f"{len(training)} training, {len(holdout)} holdout companies")

373 training, 160 holdout companies


## 2. Inventory (no results)

In [4]:
D = fc.DIRECT
freeze = json.loads((D / "freeze.json").read_text())
pd.Series({"Barbara's direct-peer pairs": freeze["counts"]["pairs"],
           "comp outside the universe (left out of v1a)": freeze["counts"]["external_pairs"],
           "comp in the universe (final_holdout_24)": freeze["counts"]["holdout_pairs"],
           "facets": f'{freeze["facet_model"]}, prompt {freeze["facet_prompt_version"]}',
           "agent loop": f'{freeze["agent_model"]}, prompt {freeze["agent_prompt_version"]}'}).to_frame("value")

,value
Barbara's direct-peer pairs,90
comp outside the universe (left out of v1a),66
comp in the universe (final_holdout_24),24
facets,"claude-cli claude-sonnet-5-5, prompt 2026-10-25k"
agent loop,"claude-cli sonnet, search sonnet, prompt 2026-11-05b"


In [5]:
nb.inputs([p.relative_to(ROOT).as_posix() for p in sorted(D.glob("*")) if p.is_file()])

,input,rows,sha256,status
0,outputs-local/v1a/10_final_holdout/direct_peers_24/agent_config.json,NaN,9ed6c8e6efac,matches seed
1,outputs-local/v1a/10_final_holdout/direct_peers_24/agent_instructions.txt,NaN,5b6bcf69f5fb,matches seed
2,outputs-local/v1a/10_final_holdout/direct_peers_24/agent_verdicts.csv,24.0,993306581422,matches seed
3,outputs-local/v1a/10_final_holdout/direct_peers_24/freeze.json,NaN,5e13d9c7b0a2,matches seed
4,outputs-local/v1a/10_final_holdout/direct_peers_24/manifest.csv,24.0,54af4132315b,matches seed


In [6]:
pd.DataFrame([{"folder": p.relative_to(paths.V1A_OUTPUTS / "10_final_holdout").as_posix(), "files": sum(1 for f in p.rglob("*") if f.is_file())}
              for p in sorted((paths.V1A_OUTPUTS / "10_final_holdout").glob("*/*")) if p.is_dir()]
             + [{"folder": "v1_holdout_runs (files)", "files": sum(1 for f in (paths.V1A_OUTPUTS / "10_final_holdout" / "v1_holdout_runs").glob("*") if f.is_file())}])

,folder,files
0,direct_peers_24/assembled_20261118T014938Z,4
1,direct_peers_24/standardized_20261118T204217Z,3
2,v1_holdout_runs/evaluation,11
3,v1_holdout_runs/graded_peer_selection,11
4,v1_holdout_runs (files),5


## 3. Checks: nothing in 01-09 used the holdout

In [7]:
import facet_uplift as fu
top = pd.read_csv(paths.path(cur["retrieval"]["candidates"]) / "candidates.csv", dtype={"target": str, "comp": str})
b3 = top[top["selected"]]
checks.add("current B3 lists: training targets and training comps only",
           set(b3["target"]) <= training and set(b3["comp"]) <= training)
j = pd.read_csv(paths.path(cur["selection"]["judgments"]), dtype={"target": str, "comp": str})
checks.add("selector judgments: training only", set(j["target"]) | set(j["comp"]) <= training)
v = pd.read_csv(paths.path(cur["agent_grading"]["run"]) / "verdicts.csv", dtype={"target": str, "comp": str})
checks.add("agent grading run: training only", set(v["target"]) | set(v["comp"]) <= training)
for name in cur["labels"]["cohorts"]:
    c = cohorts.load(name)
    checks.add(f"cohort {name}: training only", set(c["target_id"]) | set(c["comp_id"]) <= training)
for name, r in cur["analysis"]["facet_uplift"].items():
    f = pd.read_csv(paths.path(r) / "pair_features.csv", dtype={"target": str, "comp": str})
    checks.add(f"X-learner {name}: training only", set(f["target"]) | set(f["comp"]) <= training)
try:
    cohorts.load("final_holdout_24"); refused = False
except cohorts.LockedCohort:
    refused = True
checks.add("final_holdout_24 is refused without final_check", refused)
locked = []
for part in (fc.direct_peers, fc.holdout):
    try:
        part()
    except fc.Locked:
        locked.append(part.__name__)
checks.add("final_check.py refuses without final_check", locked == ["direct_peers", "holdout"])
allowed = {"direct_peer_pairs.py", "cohorts.py", "facet_uplift.py"}   # writer, lock, docstring
mentions = sorted(p.name for p in (ROOT / "v2").glob("0[1-9]_*/scripts/**/*.py")
                  if any(w in p.read_text(encoding="utf-8") for w in ("final_holdout_24", "direct_peers_24", "10_final_holdout")))
checks.add("only the writer and the lock name the locked set in 01-09", set(mentions) <= allowed, ", ".join(mentions))
checks.show()

15/15 checks pass


,check,result,detail
0,current B3 lists: training targets and training comps only,pass,
1,selector judgments: training only,pass,
2,agent grading run: training only,pass,
3,cohort dev_293: training only,pass,
4,cohort round1_36: training only,pass,
5,cohort matthew_v2_262: training only,pass,
6,X-learner human_peer: training only,pass,
7,X-learner human_rating: training only,pass,
8,X-learner selector_all: training only,pass,
9,X-learner selector_293: training only,pass,


## 4. Final check (runs only with `FINAL_CHECK = True`)

4.1 Barbara's 24 direct peers through the 293 pairs' steps (agent grade, C/S/E, B3 rank if the comp were in the
pool). 4.2 The current B3 lists of the 160 holdout targets, valued against "no matching". The selector on the
holdout needs an LLM run first (04, approval first).

In [8]:
if FINAL_CHECK:
    dp = fc.direct_peers(final_check=True)
    hv = fc.holdout(final_check=True)
    print(dp, hv, sep="\n")
    display(pd.read_csv(dp / "pairs.csv"))
    display(pd.read_csv(hv / "valuation.csv"))
else:
    print("Locked. Set FINAL_CHECK = True when the team decides the method is final.")

Locked. Set FINAL_CHECK = True when the team decides the method is final.


## 5. History (v1 record)

- **10-28 (v0 deliverable):** the frozen v0 routes (cosine, LLM-then-cosine, direct LLM, cosine-then-LLM) on all 160
  holdout companies (`holdout_split_candidates.py`, `holdout_llm_routes.py`, `holdout_batched_routes.py`,
  `holdout_evaluation.py`); Sol's cited 0-2 grades as a route (`graded_peer_selection.py`); the companion notebook
  (`build_holdout_report.py` -> `v1/notebooks/gold_and_holdout_integration.ipynb`). These results have been seen,
  so the holdout is no longer untouched for those routes; the current method (Sonnet facets, B3 variant, Sonnet
  selector) has not been run on it.
- **11-17/18 (Fiona):** Barbara's 90 direct-peer pairs: external facets and the agent loop on all 90; the 24
  in-universe pairs were standardized (11-18) and are the locked cohort here. The 66 external pairs are left out
  of v1a (no website scrape or Sonnet fill for those companies).